<a href="https://colab.research.google.com/github/mdkamrulhasan/machine_learning_concepts/blob/master/notebooks/supervised/Regression_medical_insurance_LR_kNN_DTree_feature_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### What will we cover today ?

We are going to learn how to train and test a Regression model on a real dataset. More specifically we are going to use the sklearn implementation of the following three regression models for a regression task (***diabetest severity prediction***).

1.   **Linear Regression**
2.   **kNN Regression**
3.   **Decision Tree Regressor**








[Medical insurance cost dataset](https://www.kaggle.com/datasets/mirichoi0218/insurance/)

**Summary of the dataset:**

*   **Features:** age, sex, bmi, children, smoker, region
*   **Target (y=charges)**: medical insurance charges.







---


---





## Loading necessary python packages

In [ ]:
# data processing packages
import pandas as pd

# Regression modeling package(s) (sklearn)
from sklearn.linear_model import LinearRegression

# model evaluation related packages
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# visualization
import plotly.express as px
import plotly.graph_objects as go

import matplotlib.pyplot as plt
import seaborn as sns

## Loading data and some preprocessing

In [ ]:
# Load the  dataset
data_url = 'https://raw.githubusercontent.com/mdkamrulhasan/data_mining_kdd/main/data/medical-cost/insurance.csv'
df = pd.read_csv(data_url)
df.keys()

In [ ]:
df.head()

In [ ]:
# Separating features and labels dataframes
features_df, labels_df = df[df.columns[:-1]], df['charges']

In [ ]:
# Looking at a sample of features
features_df.head()

In [ ]:
features_df.shape

In [ ]:
labels_df.head()

In [ ]:
# plotting the (bmi, target) pair values
fig = px.scatter(x=features_df['bmi'], y=labels_df.values)
fig.show()

In [ ]:
sns.scatterplot(x='bmi', y='charges', hue='smoker', data=df)
plt.title('BMI vs. Insurance Charges')

In [ ]:
sns.scatterplot(x='bmi', y='charges', data=df[df.smoker=='yes'])
plt.title('BMI vs. Insurance Charges')

We clearly see a linear relationship between the bmi feature and the disease progresson rate

In [ ]:
features_df.head()

### Q. Can we input these features to models (say Linear Regressor/kNN) as they are?

## Feature Preprocessing

In [ ]:
features_df.info()

## Two types of features: numeric vs categorical; Lets build a pipeline to automate this feature separation process.

In [ ]:
features_numeric = []
features_categorical = []
for colx in features_df.columns:
  if features_df[colx].dtype == 'object':
    #print(colx, df[colx].nunique())
    features_categorical.append(colx)
  else:
    features_numeric.append(colx)
print('numeric features:', features_numeric)
print('categorical features:', features_categorical)

## Label Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder


encoded_df = features_df.copy()

label_encoder = LabelEncoder()
for featx in features_categorical:
  encoded_df[featx] = label_encoder.fit_transform(encoded_df[featx])

In [ ]:
encoded_df.head()

## One Hot Encoding

This is little tricker than the Label Encoder.

In [ ]:
# lets check with the first categorical feature
f_categorical = features_categorical[0]
one_hot_transform = pd.get_dummies(
    features_df[f_categorical], prefix=f_categorical+'_')
one_hot_transform.head(3)

In [ ]:
# lets do for all categorical features
one_hot_dataframes = []
for featx in features_categorical:
  one_hot_dataframes.append(
      pd.get_dummies(features_df[featx], prefix=featx+'_')
  )

In [ ]:
# now merge all process dataframes
encoded_df = pd.concat([features_df[features_numeric]]
                           + one_hot_dataframes, axis=1)
encoded_df.head()

## Now we can feed these data to our models; wait; do you see any issue related to the scale of these data/features?

In [ ]:
encoded_df.min(), encoded_df.max()

## Lets scale our data

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler
# min-max scaling
feature_scaler = MinMaxScaler()
# standard scaling
# feature_scaler = StandardScaler()
encoded_df_scaled = feature_scaler.fit_transform(encoded_df.to_numpy())
df_preprocessed = pd.DataFrame(encoded_df_scaled, columns=encoded_df.columns)

In [ ]:
df_preprocessed.min(), df_preprocessed.max()

In [ ]:
df_preprocessed.head()

In [ ]:
df_preprocessed.age.min(), df_preprocessed.age.max()



---



---



## Good news: Our data is now ready to feed our models



---



---



# Modeling

Extracting features and labels as numpy matrices

In [ ]:
X, y = df_preprocessed.values, labels_df.values

Splitting data into train, test splits

In [ ]:
# test data amount (in terms of proportion)
TEST_PROP = 0.5
# Random number seed; important for experiment reproducibility
RANDOM_SEED = 0

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_PROP, random_state=RANDOM_SEED)
X_train.shape, y_train.shape, X_test.shape, y_test.shape

Model Instantiation

In [ ]:
# Create a linear regression object
regr = LinearRegression()

Model Training

In [ ]:
# Train the model using the training set
regr.fit(X_train, y_train)

In [ ]:
regr.coef_, regr.intercept_

Making Predictions and evaluation (on the traning data)

- just checking how good the model fit was on the training data.

In [ ]:
# Making predictions (training dataset)
y_pred = regr.predict(X_train)
# Estimating the Mean Squared Error (MSE)
mse_train = mean_squared_error(y_train, y_pred)
print("Mean squared error (training data): %.2f" % mse_train)

Making Predictions and evaluation (on the test data)

- This is more interesting metric as we are reporting on unseen data (by the model)

In [ ]:
# Making predictions (test dataset)
y_pred = regr.predict(X_test)
# Estimating the Mean Squared Error (MSE)
mse_test = mean_squared_error(y_test, y_pred)
print("MSE (test data): %.2f" % mse_test)

In [ ]:
lreg_results = pd.DataFrame({
  'model': ['lr'],
  'train_err': [round(mse_train, 2)],
  'test_err': [round(mse_test, 2)]
})

## kNN Regressor

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

N_NEAREST_NEIGHOR_PARAM = 10

# Model instantiation and training
reg_knn = KNeighborsRegressor(n_neighbors=N_NEAREST_NEIGHOR_PARAM)
reg_knn.fit(X_train, y_train)

# Prediction and error estimation (traing data)
y_pred = reg_knn.predict(X_train)
mse_train = mean_squared_error(y_train, y_pred)
print("MSE (train data): %.2f" % mse_train)

# Prediction and error estimation (test data)
y_pred = reg_knn.predict(X_test)
mse_test = mean_squared_error(y_test, y_pred)
print("MSE (test data): %.2f" % mse_test)

# Storing results in a dataframe
knn_results = pd.DataFrame({
  'model': ['knn'],
  'train_err': [round(mse_train, 2)],
  'test_err': [round(mse_test, 2)]
})

## Decison Tree Regressor

In [ ]:
from sklearn.tree import DecisionTreeRegressor

# Model instantiation and training
reg_dtree = DecisionTreeRegressor()
reg_dtree.fit(X_train, y_train)

# Prediction and error estimation (traing data)
y_pred = reg_dtree.predict(X_train)
mse_train = mean_squared_error(y_train, y_pred)
print("MSE (train data): %.2f" % mse_train)

# Prediction and error estimation (test data)
y_pred = reg_dtree.predict(X_test)
mse_test = mean_squared_error(y_test, y_pred)
print("MSE (test data): %.2f" % mse_test)

# Storing results in a dataframe
dtree_results = pd.DataFrame({
  'model': ['dtree'],
  'train_err': [round(mse_train, 2)],
  'test_err': [round(mse_test, 2)]
})

## Comparing model performances

In [ ]:
results = pd.concat([lreg_results, knn_results, dtree_results], axis=0)

fig = go.Figure([
    go.Bar(x=results.model, y=results.train_err, name='Training error'),
    go.Bar(x=results.model, y=results.test_err, name='Test error')
]
               )
fig.update_layout(
    title="Model comparison", yaxis_title="MSE")
fig.update_layout(
    legend=dict(
        x=0.05,
        y=0.95
    )
)
fig.show()

# Questions for you



*   Any differences have you noticed among these three model performances ?
*   Which is your preferred model and why?
*   What steps would you take to make a fair comparion among these models (model classes)?



